In [ ]:

import numpy as np
import math

# Objective function: minimize spring design cost
def objective(x):
    w, d, L = x
    return (L + 2) * w**2 * d

# Engineering constraints from the research paper
def constraints(x):
    w, d, L = x

    g1 = 1 - (d**3 * L) / (71785 * w**4)
    g2 = 1 - (140.45 * w) / (d**2 * L)
    g3 = 2 * (w + d) / 3 - 1

    g4 = (
        d * (4*d - w) /
        (w**3 * (12566*d - w))
        + 1 / (5108 * w**2) - 1
    )

    return np.array([g1, g2, g3, g4])

# Penalize designs that violate constraints
def fitness(x):
    f = objective(x)
    g = constraints(x)
    penalty = 1e5 * np.sum(np.maximum(g, 0)**2)
    return f + penalty

# Generate a Levy-flight step
def levy_flight(dim):
    beta = 1.5

    sigma = (
        math.gamma(1 + beta) *
        math.sin(math.pi * beta / 2) /
        (
            math.gamma((1 + beta) / 2)
            * beta * 2**((beta - 1) / 2)
        )
    )**(1 / beta)

    u = np.random.normal(0, sigma, dim)
    v = np.random.normal(0, 1, dim)

    return u / (np.abs(v)**(1 / beta) + 1e-12)

# Main Cuckoo Search algorithm
def cuckoo_search():
    np.random.seed(42)

    N = 25
    Pa = 0.25
    iterations = 3000

    # Lower and upper bounds: w, d, L
    lb = np.array([0.05, 0.25, 2.0])
    ub = np.array([2.0, 1.3, 15.0])

    # Initialize nests randomly
    nests = lb + (ub - lb) * np.random.rand(N, 3)
    scores = np.array([fitness(x) for x in nests])

    best_idx = np.argmin(scores)
    best = nests[best_idx].copy()
    best_score = scores[best_idx]

    for t in range(iterations):

        # Generate new solutions using Levy flights
        for i in range(N):
            step = levy_flight(3)

            new_nest = nests[i] + 0.01 * step * (ub - lb)
            new_nest = np.clip(new_nest, lb, ub)

            new_score = fitness(new_nest)
            j = np.random.randint(N)

            if new_score < scores[j]:
                nests[j] = new_nest
                scores[j] = new_score

        # Abandon the worst nests
        k = max(1, round(Pa * N))
        worst = np.argsort(scores)[-k:]

        p1 = np.random.permutation(N)
        p2 = np.random.permutation(N)

        new_nests = nests[worst] + np.random.rand() * (
            nests[p1[:k]] - nests[p2[:k]]
        )

        new_nests = np.clip(new_nests, lb, ub)

        for q, i in enumerate(worst):
            new_score = fitness(new_nests[q])

            if new_score < scores[i]:
                nests[i] = new_nests[q]
                scores[i] = new_score

        # Update the best solution
        idx = np.argmin(scores)

        if scores[idx] < best_score:
            best = nests[idx].copy()
            best_score = scores[idx]

    print("Best design (w, d, L):", best)
    print("Minimum objective value:", objective(best))
    print("Constraint values:", constraints(best))
    print(
        "All constraints satisfied:",
        np.all(constraints(best) <= 1e-6)
    )

cuckoo_search()

# Nisha 1WN24CS183

Best design (w, d, L): [0.05       0.37441887 8.55616195]
Minimum objective value: 0.009881065697795432
Constraint values: [-1.01013769e-03 -4.85460075e+00 -7.17054083e-01 -3.56410454e-05]
All constraints satisfied: True
